<a href="https://colab.research.google.com/github/JonathanA214/Study_guide/blob/main/Copy_of_CPSC483_HW3_breastcancer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CPSC 483 Homework 3

---
In this homework assignment, you'll explore the "Breast Cancer Wisconsin" dataset, which can be found as a library in both the UCI Machine Learning Repository and Scikit-learn.

**LIST COLLABORATORS HERE** (double click a text block to edit it):

# Setup

This project requires Python 3.7 or above:

In [ ]:
import sys

assert sys.version_info >= (3, 7)

Scikit-Learn ≥1.0.1 is required:

In [ ]:
from packaging import version
import sklearn

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

Let's define the default font sizes, to plot pretty figures:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=12)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=12)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

Please choose a different random seed - feel free to match the seed with your collaborators to see if you get similar results.

In [ ]:
import numpy as np
import pandas as pd
import time

seed = 22 # Change this to your favorite number
np.random.seed(seed)

We will be working with the [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) dataset, where the features describe the characteristics of the cell nuclei of a breast mass, collected from the University of Wisconsin, Madison. The goal is to determine whether the cancer cell is benign or malignant.

I am providing you with the code to download the dataset, formatting it into a Pandas DataFrame. I am also proiding you with the preprocessing pipeline, which you may use for some (not all!) of the models.

**NOTE**: The labels in this dataset are formatted such that
*   Malignant = +1
*   Benign = -1

This will help with your analysis later!



In [ ]:
from sklearn import set_config
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# fetch dataset
breast_cancer_df = load_breast_cancer(as_frame=True).frame

# Make them -1 and 1 for SVMs
# Malignant = +1, Benign = -1
breast_cancer_df["target"] = -1 * ((2 * breast_cancer_df["target"]) - 1)

# Create a Test Set
train, test = train_test_split(breast_cancer_df, test_size=0.2, random_state=seed)

# Separate out the labels
breast_cancer_train = train.drop("target", axis=1)
breast_cancer_train_labels = train['target'].copy()

# Create preprocessing pipeline (for SVMs)
preprocessing = make_pipeline(StandardScaler())

# Peak at our preprocessed data (for SVMs)
breast_cancer_processed = preprocessing.fit_transform(breast_cancer_train)
breast_cancer_processed_df = pd.DataFrame(
    breast_cancer_processed,
    columns=preprocessing.get_feature_names_out(),
    index=breast_cancer_train.index)
breast_cancer_processed_df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
567,1.774412,2.359322,1.912266,1.660686,1.486634,3.222165,3.260019,2.580670,2.191595,1.036555,...,1.913058,2.277673,2.240046,1.610056,1.434147,3.854552,3.201787,2.253309,2.032150,2.200896
295,-0.132267,-1.389240,-0.191797,-0.231329,-0.354830,-0.835512,-0.998538,-0.791615,-0.831686,-0.536907,...,-0.360381,-1.421296,-0.419472,-0.409106,-0.726349,-0.963641,-1.150876,-0.895566,-0.135710,-0.904210
91,0.314393,0.825819,0.299043,0.172141,-0.353404,-0.053598,0.270768,0.621480,-0.362232,-0.273238,...,0.001069,0.044147,-0.027441,-0.113517,-0.334759,-0.377910,0.043211,0.467253,-0.593636,-0.885377
1,1.766037,-0.338896,1.621158,1.829839,-0.871182,-0.525126,-0.045402,0.503172,-0.005448,-0.886090,...,1.759031,-0.355519,1.481276,1.845065,-0.420279,-0.460862,-0.164431,1.051448,-0.260911,0.263390
94,0.227853,0.141929,0.303086,0.109472,0.495296,0.923086,0.993087,0.959648,0.156044,-0.006719,...,0.370734,-0.220652,0.443114,0.224374,0.988545,1.018980,1.181380,1.439390,-0.116844,-0.106609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
369,2.168031,0.625087,2.199331,2.262426,0.666462,1.680630,1.927853,2.532433,0.039620,-0.211953,...,2.307368,0.037568,2.545906,2.316824,-0.168221,0.817615,0.965529,1.921656,-0.276346,0.081714
320,-1.114918,-0.710017,-1.062695,-0.948149,0.652198,0.088087,-0.290841,-0.271669,-0.264586,1.411392,...,-1.056584,-0.816040,-1.085307,-0.880342,0.317891,-0.148050,-0.414567,-0.295853,-0.504452,0.706501
527,-0.531469,-1.622650,-0.560534,-0.548006,-0.493903,-0.819265,-0.761723,-0.606283,-0.467390,-0.685132,...,-0.578073,-1.036432,-0.623869,-0.576571,-0.177223,-0.329152,-0.466237,-0.150412,0.356518,-0.462206
125,-0.109934,-0.469605,-0.176433,-0.214691,-0.649379,-0.851569,-0.953924,-0.988626,-0.749062,-0.568263,...,-0.191978,-0.327559,-0.239191,-0.296301,-0.784863,-0.787605,-0.939226,-1.001755,-0.922931,-0.689300


# Part 1 - Training Support Vector Machines and Decision Trees

Here, we will explore the different techniques to learn support vector machines used in class, and compare/contrast them in terms of computation time and validation performance.

## Exercise 1: Linear SVMs with `LinearSVC`

Learn a linear support vector machine model using `LinearSVC`. Use `GridSearchCV` with 5 folds to find the best `C` value, and use one of the following metrics that you deem appropriate for this setting:
*   Accuracy (`scoring='accuracy'`)
*   Precision (`scoring='precision'`)
*   Recall (`scoring='recall'`)
*   F1 (`scoring='f1'`)

Hints:
*   Ensure you use the scaled data for SVMs, as SVMs are sensitive to feature scaling.
*   You can start by testing `C` values in powers of 10 (0.001, 0.01, 0.1, 1, 10, etc.) and fine tuning through trial and error.
*  Although not required for Exercise 1, feel free to try out either the primal or dual formulations. You'll find that since the dataset is small, the computational differences between them is minimal.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Insert your code here
pass

In [ ]:
# Check your 5-fold CV performance metric here (change these variables to show your best model)
print("Performance: ", grid_search_linearsvm.best_score_)
print("Best hyperparameters: ", grid_search_linearsvm.best_params_)

NameError: name 'grid_search_linearsvm' is not defined

Justify why you chose your performance measure for this problem.

**Your Answer:**

## Exercise 2: SVMs with a Polynomial Kernel

Use GridSearch to find the best parameters using the `SVC` class with a polynomial kernel. Use `GridSearchCV` with 5 folds to find the best combination of hyperparameters, and use the same scoring metric you chose from before.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Insert your code here
pass

In [ ]:
# Check your 5-fold CV performance metric here (change these variables to show your best model)
print("Performance: ", grid_search_polysvm.best_score_)
print("Best hyperparameters: ", grid_search_polysvm.best_params_)

## Exercise 3: SVMs with a Gaussian RBF Kernel

Use GridSearch to find the best parameters using the `SVC` class with a Gaussian RBF kernel. Use `GridSearchCV` with 5 folds to find the best combination of hyperparameters, and use the same scoring metric you chose from before.

*Hint:* The default for `gamma` is `1/n` for `n` features. Start with that, and vary this parameter lower/higher.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Insert your code here
pass

In [ ]:
# Check your 5-fold CV performance metric here (change these variables to show your best model)
print("Performance: ", grid_search_rbfsvm.best_score_)
print("Best hyperparameters: ", grid_search_rbfsvm.best_params_)

## Exercise 4: Classification Trees with `DecisionTreeClassifier`

If done correctly, you'll see that the SVMs above did a fantastic job in validation.

Now for the *real* challenge: match the performance of the SVM classifiers with a classification tree! We'll say you succeed if you are able to get a vaidation performance that is around $x - 0.05$ (or better!), where $x$ is your best-performing SVM model from exercises 1-3 (if you don't quite make it, it's OK! Just try your best here.)

Like before, use GridSearch to find the best parameters using the `DecisionTreeClassifier` class. Use `GridSearchCV` with 5 folds to find the best combination of hyperparameters, and use the same scoring metric you chose from before.

*Hint:*:

*   Ensure you set the `random_state` in your decision tree classifier to the random seed you chose at the top of this assignment! Otherwise, every run you do may have different results.
*   You will find yourself getting overwhelmed and waiting too long if you test out every single hyperparameter combination at first! Try a few, then make a judgement call to test other hyperparameters (perhaps see if you're overfitting or underfitting), then repeat until you start increasing your validation performance.
*   Note that you also don't need to preprocess your data!



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# Insert your code here
pass

In [ ]:
# Check your 5-fold CV performance metric here (change these variables to show your best model)
print("Performance: ", grid_search_tree.best_score_)
print("Best hyperparameters: ", grid_search_tree.best_params_)

In [ ]:
from sklearn.tree import plot_tree
# Print out your tree (change these variables to show your best model)
tree = grid_search_tree.best_estimator_['tree']

plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=breast_cancer_train.columns, class_names=["benign", "malignant"])
plt.show()

# Part 2 - Ensemble Methods

We will see how ensemble methods can help improve the performance of our decision tree from exercise 4.

## Exercise 5: Adaboost

Use Adaboost with decision stumps to learn a classifier that is better than at least one of your SVM methods from exercises 1-3 in terms of your chosen performance metric. Use `GridSearchCV` with 5 folds to find the best combination of hyperparameters, and use the same scoring metric you chose from before.

Ensure you set the random state for both the `DecisionTreeClassifier` and `AdaBoostClassifier`!


In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Insert your code here
pass

In [ ]:
# Check your 5-fold CV performance metric here (change these variables to show your best model)
print("Performance: ", grid_search_adaboost.best_score_)
print("Best hyperparameters: ", grid_search_adaboost.best_params_)

## Exercise 6: Random Forest

Learn a random forest classifier that, at minimum, is closer to the performance of your best model so far than the decision tree from Exercise 4. Use `GridSearchCV` with 5 folds to find the best combination of hyperparameters, and use the same scoring metric you chose from before.

Ensure you set the random state!

*Hint:* You may want to refer to your solution for Exercise 4 for any potentially good hyperparameters for your forest!


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Insert your code here
pass

In [ ]:
# Check your 5-fold CV performance metric here (change these variables to show your best model)
print("Performance: ", grid_search_forest.best_score_)
print("Best hyperparameters: ", grid_search_forest.best_params_)

## Exercise 7: Your Final Decision

You have now walked through the real thought process of a machine learning engineer in tackling this dataset from a variety of different angles!

Now for your ultimate decision: based on exercises 1-6, observing the performance of each model AND considering the tradeoffs between interpretability and accuracy, make a final recommendation for a model. Explain why you are choosing that model in the code cell below.

**Your Answer:**

Then, with that final model, compute the test performance on your chosen metric.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Change these values to your model and chosen metric
final_model = None
metric = None # Insert the sklearn *_score function here

breast_cancer_test = test.drop("target", axis=1)
breast_cancer_test_labels = test['target'].copy()

final_predictions = final_model.predict(breast_cancer_test)

final_metric = metric(breast_cancer_test_labels, final_predictions)
print("Test Performance:", final_metric)